## **CLI Tool Development Exercises**  
**Advanced CLI Features** including **subcommands, environment variables, logging, configuration files, and testing**.  
  
🎯 **Objective:**  
Build a **CLI-based Task Manager** with **subcommands, logging, environment variables, and testing**.  

---

### **🔹 Requirements**
✅ **Core Features**:  
- Users can **add a task**, **list tasks**, and **delete a task**.  
- Tasks are stored in a **JSON file (`tasks.json`)**.  

✅ **Advanced Features**:  
- Implement **subcommands (`add`, `list`, `delete`)** using `argparse`.  
- **Log all actions** in a `task_manager.log` file.  
- Use an **environment variable (`TASKS_FILE_PATH`)** to specify the task file.  
- Implement **unit tests** for `add_task()` and `delete_task()`.  

---

### **📌 Expected Folder Structure**
```plaintext
advanced_cli_task_manager/
│── task_manager/          # CLI Tool Source Code
│   │── __init__.py
│   │── cli.py             # CLI Entry Point
│   │── core.py            # Core Logic
│   │── logger.py          # Logging Setup
│   │── config.py          # Configuration Management
│── tests/                 # Unit Tests
│   │── test_core.py
│── tasks.json             # JSON File for Storing Tasks
│── requirements.txt       # Dependencies
│── README.md              # Documentation
```

---

### **Step 1: Implement CLI Tool (`cli.py`)**
```python
import argparse
from task_manager.core import add_task, list_tasks, delete_task
from task_manager.logger import setup_logger

logger = setup_logger()

def main():
    parser = argparse.ArgumentParser(description="CLI Task Manager")
    subparsers = parser.add_subparsers(dest="command", help="Available commands")

    # Add Task
    # ...

    # List Tasks
    # ...

    # Delete Task
    # ...

    # Actions
    args = parser.parse_args()

    # ...

if __name__ == "__main__":
    main()
```

---

### **📌 Step 2: Implement Core Logic (`core.py`)**
```python
import json
import os
from task_manager.logger import setup_logger

TASKS_FILE = os.getenv("TASKS_FILE_PATH", "tasks.json")
logger = setup_logger()

def load_tasks():
    # ...

def save_tasks(tasks):
    # ...

def add_task(description, priority):
    # ...

def list_tasks():
    # ...

def delete_task(task_id):
    # ....
   
```

---

### **📌 Step 3: Implement Logging (`logger.py`)**
```python
import logging
import os

def setup_logger(log_file="task_manager.log"):
    log_directory = "logs"
    # ...

    return logger
```

---

### **📌 Step 4: Implement Testing (`test_core.py`)**
```python
import unittest
from task_manager.core import add_task, load_tasks, delete_task

class TestTaskManager(unittest.TestCase):
    def test_add_task(self):
        # ...

    def test_delete_task(self):
        # ...

if __name__ == "__main__":
    unittest.main()
```


## ✅ Solution

Le projet complet a été créé dans le dossier `advanced_cli_task_manager/` avec la structure suivante :

```
advanced_cli_task_manager/
├── task_manager/
│   ├── __init__.py
│   ├── cli.py
│   ├── core.py
│   ├── logger.py
│   └── config.py
├── tests/
│   └── test_core.py
├── requirements.txt
└── README.md
```

### Utilisation
```bash
cd advanced_cli_task_manager
python -m task_manager.cli add "Buy milk" --priority high
python -m task_manager.cli list
python -m task_manager.cli delete 1
python -m unittest tests/test_core.py
```

Le code de chaque fichier est détaillé ci-dessous.

### `task_manager/cli.py`

In [ ]:
%%writefile task_manager/cli.py
import argparse
from task_manager.core import add_task, list_tasks, delete_task
from task_manager.logger import setup_logger

logger = setup_logger()


def main():
    parser = argparse.ArgumentParser(description="CLI Task Manager")
    subparsers = parser.add_subparsers(dest="command", help="Available commands")

    # Add Task
    add_parser = subparsers.add_parser("add", help="Add a new task")
    add_parser.add_argument("description", help="Description of the task")
    add_parser.add_argument(
        "--priority", choices=["low", "medium", "high"], default="medium",
        help="Priority of the task (default: medium)"
    )

    # List Tasks
    subparsers.add_parser("list", help="List all tasks")

    # Delete Task
    delete_parser = subparsers.add_parser("delete", help="Delete a task by id")
    delete_parser.add_argument("task_id", type=int, help="ID of the task to delete")

    args = parser.parse_args()

    if args.command == "add":
        task = add_task(args.description, args.priority)
        print(f"Added task #{task['id']}: {task['description']} (priority={task['priority']})")

    elif args.command == "list":
        tasks = list_tasks()
        if not tasks:
            print("No tasks found.")
        for t in tasks:
            status = "✔" if t["done"] else "✘"
            print(f"[{status}] #{t['id']} {t['description']} (priority={t['priority']})")

    elif args.command == "delete":
        if delete_task(args.task_id):
            print(f"Deleted task #{args.task_id}")
        else:
            print(f"Task #{args.task_id} not found")

    else:
        parser.print_help()


if __name__ == "__main__":
    main()


### `task_manager/core.py`

In [ ]:
%%writefile task_manager/core.py
import json
import os

from task_manager.logger import setup_logger
from task_manager.config import TASKS_FILE

logger = setup_logger()


def load_tasks():
    """Load tasks from the JSON file. Returns an empty list if the file doesn't exist."""
    if not os.path.exists(TASKS_FILE):
        return []
    try:
        with open(TASKS_FILE, "r") as f:
            return json.load(f)
    except json.JSONDecodeError:
        logger.error("Tasks file is corrupted, starting with an empty list.")
        return []


def save_tasks(tasks):
    """Save the list of tasks to the JSON file."""
    with open(TASKS_FILE, "w") as f:
        json.dump(tasks, f, indent=4)


def add_task(description, priority="medium"):
    """Add a new task and return it."""
    tasks = load_tasks()
    new_id = (max((t["id"] for t in tasks), default=0)) + 1
    task = {
        "id": new_id,
        "description": description,
        "priority": priority,
        "done": False,
    }
    tasks.append(task)
    save_tasks(tasks)
    logger.info(f"Added task {new_id}: {description} (priority={priority})")
    return task


def list_tasks():
    """Return all tasks."""
    tasks = load_tasks()
    logger.info(f"Listed {len(tasks)} task(s)")
    return tasks


def delete_task(task_id):
    """Delete a task by id. Returns True if deleted, False if not found."""
    tasks = load_tasks()
    remaining = [t for t in tasks if t["id"] != task_id]
    if len(remaining) == len(tasks):
        logger.warning(f"Tried to delete non-existent task {task_id}")
        return False
    save_tasks(remaining)
    logger.info(f"Deleted task {task_id}")
    return True


### `task_manager/logger.py`

In [ ]:
%%writefile task_manager/logger.py
import logging
import os


def setup_logger(log_file="task_manager.log"):
    """Configure and return a logger that writes to logs/<log_file>."""
    log_directory = "logs"
    os.makedirs(log_directory, exist_ok=True)
    log_path = os.path.join(log_directory, log_file)

    logger = logging.getLogger("task_manager")
    logger.setLevel(logging.INFO)

    # Avoid adding duplicate handlers if setup_logger() is called multiple times
    if not logger.handlers:
        handler = logging.FileHandler(log_path)
        formatter = logging.Formatter(
            "%(asctime)s - %(levelname)s - %(message)s"
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger


### `task_manager/config.py`

In [ ]:
%%writefile task_manager/config.py
import os

# Path to the JSON file used to store tasks.
# Can be overridden with the TASKS_FILE_PATH environment variable.
TASKS_FILE = os.getenv("TASKS_FILE_PATH", "tasks.json")


### `tests/test_core.py`

In [ ]:
%%writefile tests/test_core.py
import os
import unittest

os.environ["TASKS_FILE_PATH"] = "test_tasks.json"

from task_manager.core import add_task, load_tasks, delete_task


class TestTaskManager(unittest.TestCase):
    def setUp(self):
        # Start each test with a clean tasks file
        if os.path.exists("test_tasks.json"):
            os.remove("test_tasks.json")

    def tearDown(self):
        if os.path.exists("test_tasks.json"):
            os.remove("test_tasks.json")

    def test_add_task(self):
        task = add_task("Write unit tests", priority="high")
        tasks = load_tasks()
        self.assertEqual(len(tasks), 1)
        self.assertEqual(tasks[0]["description"], "Write unit tests")
        self.assertEqual(tasks[0]["priority"], "high")
        self.assertEqual(task["id"], 1)

    def test_delete_task(self):
        task = add_task("Temporary task")
        deleted = delete_task(task["id"])
        self.assertTrue(deleted)
        self.assertEqual(load_tasks(), [])

        # Deleting a non-existent task should return False
        self.assertFalse(delete_task(999))


if __name__ == "__main__":
    unittest.main()
